In [6]:
import numpy as np
import pandas as pd
from transformers import BertTokenizerFast, BertForQuestionAnswering
import datasets
from functools import partial
from ruler.data.load_data import load_data, split
from ruler.models.evaluate_utils import data_to_qa, tokenize
import torch
import tqdm
import numpy as np
from ruler.models.grokfast import gradfilter_ema
from torch.utils.data import DataLoader
from evaluate import evaluator
from ruler.models.finetune_QA_manual import format_answers
from ruler.models import grokfast
from sklearn.metrics import f1_score, classification_report, confusion_matrix

In [3]:
data = load_data('../data/interim/training_Data/')
print(f'data len: {len(data)}')

discarding 0 rules
discarding 0 rules
discarding 0 rules
discarding 21 rules
discarding 0 rules
discarding 21 rules
date len: 17359


In [27]:
data[0]

{'ap_id': 'https://kbin.social/m/pcgaming@lemmy.ca/t/343670/-/comment/1642205',
 'applied_rule_n': 2,
 'applied_rule_text': 'No Spam or Porn.',
 'community': {'actor_id': 'https://lemmy.ca/c/pcgaming',
  'description': "For PC gaming news and discussion.\n[PCGamingWiki](https://www.pcgamingwiki.com/wiki/Home)\n\n***Rules:***\n\n0. Be Respectful.\n1. No Spam or Porn.\n2. No Advertising.\n3. No Memes.\n4. No Tech Support.\n5. No questions about buying/building computers.\n6. No game suggestions, friend requests, surveys, or begging.\n7. No Let's Plays, streams, highlight reels/montages, random videos or shorts.\n8. No off-topic posts/comments.\n9. Use the original source, no clickbait titles, no duplicates. \n(Submissions should be from the original source if possible, unless from paywalled or non-english sources.\nIf the title is clickbait or lacks context you may **lightly** edit the title.) ",
  'name': 'pcgaming',
  'nsfw': False,
  'rules': {1: 'Be Respectful.',
   2: 'No Spam or Po

In [13]:
comms = pd.DataFrame([(i['instance'], i['community']['actor_id'], i['removed']) for i in data], columns = ['instance', 'community', 'removed']).drop_duplicates()

In [14]:
comms.head()

,instance,community,removed
0,l.vidja.social,https://lemmy.ca/c/pcgaming,True
2,l.vidja.social,https://ttrpg.network/c/rpg,True
8,l.vidja.social,https://lemmy.world/c/3dprinting,True
11,lemmy.run,https://lemmy.world/c/nostupidquestions,True
17,lemmy.run,https://lemmy.world/c/reddit,True


In [16]:
positive_communities = set(comms[~comms.removed].community.unique())

In [9]:
import json
with open(r'D:\PycharmProjects\lemmymod\data\raw\communities\instances_to_local_communities.json', encoding='utf-8') as f:
    comms_scraped=json.load(f)

In [10]:
comms_scraped_df = pd.DataFrame([(i, vv['community']['actor_id']) for i, v in comms_scraped.items() for vv in v], columns = ['instance', 'community'])

In [11]:
comms_scraped_df.head()

,instance,community
0,7c4tx1.lem.rocks,https://7c4tx1.lem.rocks/c/test
1,7c4tx1.lem.rocks,https://7c4tx1.lem.rocks/c/test8
2,7c4tx1.lem.rocks,https://7c4tx1.lem.rocks/c/fake2
3,0xdd.org.ru,https://0xdd.org.ru/c/sandbox
4,0xdd.org.ru,https://0xdd.org.ru/c/games


In [24]:
comms_scraped_df.community.nunique()

26497

In [17]:
positive_communities.difference(comms_scraped_df.community.unique())

{'https://lemmy.blahaj.zone/c/nonvoters',
 'https://lemmy.ml/c/comics',
 'https://lemmy.ml/c/memes',
 'https://lemmy.ml/c/palestine',
 'https://lemmy.ml/c/privacy',
 'https://lemmy.ml/c/programming',
 'https://lemmy.ml/c/shitposting',
 'https://lemmy.ml/c/snoocalypse',
 'https://lemmy.run/c/india',
 'https://lemmy.world/c/technology',
 'https://lemmy.world/c/weedtime',
 'https://lemmy.zip/c/technology',
 'https://lemmynsfw.com/c/asklemmynsfw',
 'https://midwest.social/c/usa',
 'https://sh.itjust.works/c/whitepeopletwitter'}

In [31]:
positive_communities.difference(comms_scraped_df.community.unique())

{'https://lemmy.blahaj.zone/c/nonvoters',
 'https://lemmy.ml/c/comics',
 'https://lemmy.ml/c/memes',
 'https://lemmy.ml/c/palestine',
 'https://lemmy.ml/c/privacy',
 'https://lemmy.ml/c/programming',
 'https://lemmy.ml/c/shitposting',
 'https://lemmy.ml/c/snoocalypse',
 'https://lemmy.run/c/india',
 'https://lemmy.world/c/technology',
 'https://lemmy.world/c/weedtime',
 'https://lemmy.zip/c/technology',
 'https://lemmynsfw.com/c/asklemmynsfw',
 'https://midwest.social/c/usa',
 'https://sh.itjust.works/c/whitepeopletwitter'}

In [26]:
len(positive_communities.difference(comms_scraped_df.community.unique()))

15

In [39]:
comms_scraped_df[(comms_scraped_df.instance.apply(lambda x: 'lemmy.world' in x)) & (comms_scraped_df.community.apply(lambda x: "hnology" in x))]

,instance,community
18860,lemmy.world,https://lemmy.world/c/languagetechnology
20934,lemmy.world,https://lemmy.world/c/microntechnology
20940,lemmy.world,https://lemmy.world/c/micron_technology
22344,lemmy.world,https://lemmy.world/c/technologyhotnews
27412,lemmy.world,https://lemmy.world/c/cryptotechnology


In [25]:
all_communities = set(comms.community.unique())
all_communities.difference(comms_scraped_df.community.unique())

{'https://derp.foo/c/hackernews',
 'https://feddit.de/c/europe',
 'https://fedia.io/m/android',
 'https://hexbear.net/c/anti_cishet_aktion',
 'https://hexbear.net/c/mentalhealth',
 'https://hexbear.net/c/strugglesession',
 'https://hexbear.net/c/transenby_liberation',
 'https://lemmy.blahaj.zone/c/nonvoters',
 'https://lemmy.dbzer0.com/c/fediverse_vs_disinfo',
 'https://lemmy.fmhy.ml/c/singularity',
 'https://lemmy.ml/c/collapse',
 'https://lemmy.ml/c/comics',
 'https://lemmy.ml/c/memes',
 'https://lemmy.ml/c/palestine',
 'https://lemmy.ml/c/politics',
 'https://lemmy.ml/c/privacy',
 'https://lemmy.ml/c/programming',
 'https://lemmy.ml/c/shitposting',
 'https://lemmy.ml/c/snoocalypse',
 'https://lemmy.run/c/india',
 'https://lemmy.world/c/combatfootage',
 'https://lemmy.world/c/conservativememes2',
 'https://lemmy.world/c/mmababes',
 'https://lemmy.world/c/pics',
 'https://lemmy.world/c/technology',
 'https://lemmy.world/c/testcomm',
 'https://lemmy.world/c/weedtime',
 'https://lemmy.w

In [47]:
missing_communities = all_communities.difference(comms_scraped_df.community.unique())
missing_communities.discard('https://fedia.io/m/android')

In [49]:
len(missing_communities)

37

In [50]:
from collections import defaultdict
from urllib.parse import urlparse
missing_community_mappings = defaultdict(list)
for community in missing_communities:
    missing_community_mappings[urlparse(community).netloc].append(urlparse(community).path.split('/c/')[1])

In [51]:
with open('../data/interim/missing_communities.json', 'w+', encoding='utf-8') as f:
    json.dump(missing_community_mappings, f)

In [36]:
from urllib.parse import urlparse
comms['inferred_instance'] = comms.community.apply(lambda x: urlparse(x).netloc)
comms['inferred_community'] = comms.community.apply(lambda x: urlparse(x).path.split('/c/')[1] if urlparse(x).path.startswith('/c/') else None)

In [37]:
comms.head()

,instance,community,removed,inferred_instance,inferred_community
0,l.vidja.social,https://lemmy.ca/c/pcgaming,True,lemmy.ca,pcgaming
2,l.vidja.social,https://ttrpg.network/c/rpg,True,ttrpg.network,rpg
8,l.vidja.social,https://lemmy.world/c/3dprinting,True,lemmy.world,3dprinting
11,lemmy.run,https://lemmy.world/c/nostupidquestions,True,lemmy.world,nostupidquestions
17,lemmy.run,https://lemmy.world/c/reddit,True,lemmy.world,reddit


In [38]:
comms[comms.inferred_community.isna()] # this is actually not lemmy

,instance,community,removed,inferred_instance,inferred_community
9786,sh.itjust.works,https://fedia.io/m/android,True,fedia.io,None


In [40]:
comms = comms[~comms.inferred_community.isna()]

In [44]:
ideal_communities = {instance: g.inferred_community.tolist()
for instance, g in comms[['inferred_instance', 'inferred_community']].drop_duplicates().groupby('inferred_instance')}

In [45]:
ideal_communities

{'ani.social': ['meta',
  'thiccmoe',
  'meganemoe',
  'dungeonmeshi',
  'fangmoe',
  'manga'],
 'aussie.zone': ['news', 'environment'],
 'baraza.africa': ['main'],
 'beehaw.org': ['politics', 'news'],
 'civilloquy.com': ['worldnews'],
 'derp.foo': ['hackernews'],
 'dormi.zone': ['warframe'],
 'dubvee.org': ['news'],
 'fanaticus.social': ['realmadrid'],
 'feddit.de': ['europe'],
 'feddit.nl': ['notjustbikes'],
 'feddit.org': ['europe'],
 'feddit.uk': ['murderedbywords',
  'unitedkingdom',
  'badrealestate',
  'clevercomebacks',
  'uk_politics'],
 'hexbear.net': ['the_dunk_tank',
  'history',
  'politics',
  'neurodiverse',
  'news',
  'covid',
  'askchapo',
  'chapotraphouse',
  'games',
  'urbanism',
  'food',
  'videos',
  'chat',
  'vegan',
  'christianity',
  'dredge_tank',
  'transenby_liberation',
  'furry',
  'strugglesession',
  'em_poc',
  'technology',
  'ama',
  'fitness',
  'memes',
  'labour',
  'bloomer',
  'movies',
  'antifascism',
  'acab',
  'anti_cishet_aktion',
  'm

In [39]:
for d in data:
    if len(d['community']['rules'])>20:
        raise Exception(f"Community rules too long: {len(d['community']['rules'])}")

In [40]:
print(f"data looks like this: \n{data[0]}")

data looks like this: 
{'ap_id': 'https://kbin.social/m/pcgaming@lemmy.ca/t/343670/-/comment/1642205', 'applied_rule_n': 2, 'applied_rule_text': 'No Spam or Porn.', 'community': {'actor_id': 'https://lemmy.ca/c/pcgaming', 'description': "For PC gaming news and discussion.\n[PCGamingWiki](https://www.pcgamingwiki.com/wiki/Home)\n\n***Rules:***\n\n0. Be Respectful.\n1. No Spam or Porn.\n2. No Advertising.\n3. No Memes.\n4. No Tech Support.\n5. No questions about buying/building computers.\n6. No game suggestions, friend requests, surveys, or begging.\n7. No Let's Plays, streams, highlight reels/montages, random videos or shorts.\n8. No off-topic posts/comments.\n9. Use the original source, no clickbait titles, no duplicates. \n(Submissions should be from the original source if possible, unless from paywalled or non-english sources.\nIf the title is clickbait or lacks context you may **lightly** edit the title.) ", 'name': 'pcgaming', 'nsfw': False, 'rules': {1: 'Be Respectful.', 2: 'No S

In [41]:
train, test = split(data, random_state=42, strategy='stratified')
print(f"len train: {len(train)}, test: {len(test)}")

len train: 13887, test: 3472


In [42]:
train[0]

{'ap_id': 'https://lemmy.world/comment/10484388',
 'applied_rule_n': 1,
 'applied_rule_text': 'Be civil',
 'community': {'actor_id': 'https://lemmy.world/c/politicalmemes',
  'description': 'Welcome to politcal memes! \n\nThese are our rules:\n\n::: spoiler Be civil\nJokes are okay, but don’t intentionally harass or disturb any member of our community. Sexism, racism and bigotry are not allowed. Good faith argumentation only. No posts discouraging people to vote or shaming people for voting. \n:::\n\n::: spoiler No misinformation\nDon’t post any intentional misinformation. When asked by mods, provide sources for any claims you make. \n:::\n\n::: spoiler Posts should be memes\nRandom pictures do not qualify as memes. Relevance to politics is required.\n:::\n\n::: spoiler No bots, spam or self-promotion\nFollow instance rules, ask for your bot to be allowed on this community.\n:::\n\n',
  'name': 'politicalmemes',
  'nsfw': False,
  'rules': {1: 'Be civil',
   2: 'No misinformation',
   

In [43]:
custom_tokens = True
custom_rule_tokens = False
qa = list(map(partial(data_to_qa, custom_tokens=custom_tokens, custom_rule_tokens=custom_rule_tokens), train))


In [44]:
for d in qa:
    if len(d['entry']['rules'])>20:
        raise Exception(f"Community rules too long: {len(d['entry']['rules'])}")

In [45]:
qa[0]

{'question': 'Check if in the following message there is unsafe content according to the specified policy and categories:',
 'context': '\n    [BOR]\n    0. Safe\n1. Be civil\n2. No misinformation\n3. Posts should be memes\n4. No bots, spam or self-promotion\n[EOR]\n\n[BOC]\nHey, mod that removed this: Go fuck yourself.\n[EOC]\n',
 'answers': {'rule_number': 1,
  'text': 'Be civil',
  'removed': True,
  'start_char': 23,
  'end_char': 35,
  'rule_boundaries': {'0': (15, 23),
   '1': (23, 35),
   '2': (35, 56),
   '3': (56, 81),
   '4': (81, 116)}},
 'entry': {'ap_id': 'https://lemmy.world/comment/10484388',
  'applied_rule_n': 1,
  'applied_rule_text': 'Be civil',
  'community': {'actor_id': 'https://lemmy.world/c/politicalmemes',
   'description': 'Welcome to politcal memes! \n\nThese are our rules:\n\n::: spoiler Be civil\nJokes are okay, but don’t intentionally harass or disturb any member of our community. Sexism, racism and bigotry are not allowed. Good faith argumentation only. N

In [46]:
train[0]

{'ap_id': 'https://lemmy.world/comment/10484388',
 'applied_rule_n': 1,
 'applied_rule_text': 'Be civil',
 'community': {'actor_id': 'https://lemmy.world/c/politicalmemes',
  'description': 'Welcome to politcal memes! \n\nThese are our rules:\n\n::: spoiler Be civil\nJokes are okay, but don’t intentionally harass or disturb any member of our community. Sexism, racism and bigotry are not allowed. Good faith argumentation only. No posts discouraging people to vote or shaming people for voting. \n:::\n\n::: spoiler No misinformation\nDon’t post any intentional misinformation. When asked by mods, provide sources for any claims you make. \n:::\n\n::: spoiler Posts should be memes\nRandom pictures do not qualify as memes. Relevance to politics is required.\n:::\n\n::: spoiler No bots, spam or self-promotion\nFollow instance rules, ask for your bot to be allowed on this community.\n:::\n\n',
  'name': 'politicalmemes',
  'nsfw': False,
  'rules': {1: 'Be civil',
   2: 'No misinformation',
   

In [47]:
examples = datasets.Dataset.from_list(qa)


In [48]:
examples[0]

{'question': 'Check if in the following message there is unsafe content according to the specified policy and categories:',
 'context': '\n    [BOR]\n    0. Safe\n1. Be civil\n2. No misinformation\n3. Posts should be memes\n4. No bots, spam or self-promotion\n[EOR]\n\n[BOC]\nHey, mod that removed this: Go fuck yourself.\n[EOC]\n',
 'answers': {'end_char': 35,
  'removed': True,
  'rule_boundaries': {'0': [15, 23],
   '1': [23, 35],
   '10': None,
   '11': None,
   '12': None,
   '13': None,
   '14': None,
   '15': None,
   '2': [35, 56],
   '3': [56, 81],
   '4': [81, 116],
   '5': None,
   '6': None,
   '7': None,
   '8': None,
   '9': None,
   '97': None,
   '98': None,
   '99': None},
  'rule_number': 1,
  'start_char': 23,
  'text': 'Be civil'},
 'entry': {'ap_id': 'https://lemmy.world/comment/10484388',
  'applied_rule_n': 1,
  'applied_rule_text': 'Be civil',
  'community': {'actor_id': 'https://lemmy.world/c/politicalmemes',
   'description': 'Welcome to politcal memes! \n\nThes

In [49]:

tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
if custom_tokens:
    tokenizer.add_special_tokens(
        {"additional_special_tokens": ["[BOR]", "[EOR]", "[BOC]", "[EOC]"]}
    )
if custom_rule_tokens:
    tokenizer.add_special_tokens(
        {"additional_special_tokens": [f"[RULE{rule_n}]" for rule_n in
                                       range(max(len(entry['rules'] for entry in qa)) + 1)]}
    )
model = BertForQuestionAnswering.from_pretrained("bert-base-uncased")
model.resize_token_embeddings(len(tokenizer))

inputs = tokenize(examples, tokenizer)


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [50]:
inputs[0]

Encoding(num_tokens=512, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [51]:
idx = 121
answer = examples[idx]['answers']
rule_token_boundaries = inputs['rule_token_boundaries'][idx]

In [52]:
example = examples[idx]

In [53]:
print(example['context'])


    [BOR]
    0. Safe
1. All posts must include links to the subject matter, and no identifying information should be redacted.
2. If your source is a reactionary website, please use  [archive.is](http://archive.is/)  instead of linking directly.
3. No sectarianism.
4. TERF/SWERFs Not Welcome
5. No ableism of any kind (that includes stuff like libt*rd)
6. Do not post fellow hexbears.
7. Do not individually target other instances admins or moderators.
8. The subject of a post cannot be low hanging fruit, that is comments/posts made by a private person that have low amount of upvotes/likes/views.
9. if you post ironic rage bait im going to make a personal visit to your house to make sure you never make this mistake again
[EOR]

[BOC]
Touché on dick whistle (I read it more as dickwhistler and saw it as a stand-in for cocksucker, but when you flip it to whistle dick, the actual meaning becomes clear and I like it), and good job on being gay. Its still soy because its an archaic non-swear 

In [54]:
for rule_n, (s, e) in filter(lambda x: x[1], example['answers']['rule_boundaries'].items()):
    print(rule_n, s, e, example['context'][s:e])

0 15 23 0. Safe

1 23 129 1. All posts must include links to the subject matter, and no identifying information should be redacted.

2 129 248 2. If your source is a reactionary website, please use  [archive.is](http://archive.is/)  instead of linking directly.

3 248 268 3. No sectarianism.

4 268 295 4. TERF/SWERFs Not Welcome

5 295 356 5. No ableism of any kind (that includes stuff like libt*rd)

6 356 388 6. Do not post fellow hexbears.

7 388 456 7. Do not individually target other instances admins or moderators.

8 456 603 8. The subject of a post cannot be low hanging fruit, that is comments/posts made by a private person that have low amount of upvotes/likes/views.

9 603 730 9. if you post ironic rage bait im going to make a personal visit to your house to make sure you never make this mistake again



In [55]:
input = inputs[idx]

In [56]:
for rule_n, (s, e) in filter(lambda x: x[1], inputs['rule_token_boundaries'][idx].items()):
    print(rule_n, s, e, input.tokens[s:e])

0 21 24 ['0', '.', 'safe']
1 24 46 ['1', '.', 'all', 'posts', 'must', 'include', 'links', 'to', 'the', 'subject', 'matter', ',', 'and', 'no', 'identifying', 'information', 'should', 'be', 'red', '##act', '##ed', '.']
2 46 79 ['2', '.', 'if', 'your', 'source', 'is', 'a', 'reaction', '##ary', 'website', ',', 'please', 'use', '[', 'archive', '.', 'is', ']', '(', 'http', ':', '/', '/', 'archive', '.', 'is', '/', ')', 'instead', 'of', 'linking', 'directly', '.']
3 79 86 ['3', '.', 'no', 'sect', '##arian', '##ism', '.']
4 86 96 ['4', '.', 'ter', '##f', '/', 'sw', '##er', '##fs', 'not', 'welcome']
5 96 114 ['5', '.', 'no', 'able', '##ism', 'of', 'any', 'kind', '(', 'that', 'includes', 'stuff', 'like', 'li', '##bt', '*', 'rd', ')']
6 114 125 ['6', '.', 'do', 'not', 'post', 'fellow', 'he', '##x', '##be', '##ars', '.']
7 125 139 ['7', '.', 'do', 'not', 'individually', 'target', 'other', 'instances', 'ad', '##mins', 'or', 'moderator', '##s', '.']
8 139 175 ['8', '.', 'the', 'subject', 'of', 'a', 

In [57]:
count = 0
for doc in inputs.input_ids:
    if(doc>0).sum()> tokenizer.model_max_length: 
        count+=1
print("number of truncated docs: ",count)

number of truncated docs:  0


In [58]:
from transformers import BertTokenizerFast, BertForQuestionAnswering
import datasets
from functools import partial
from ruler.data.load_data import load_data, split, data_to_qa, tokenize
import torch
import tqdm

from ruler.models.grokfast import gradfilter_ema
from torch.utils.data import DataLoader
from evaluate import evaluator
from ruler.models.finetune_QA_manual import format_answers
from ruler.models import grokfast

data = load_data()
print(f'date len: {len(data)}')

print(f"data looks like this: \n{data[0]}")
train, test = split(data, random_state=42, strategy='stratified')
print(f"len train: {len(train)}, test: {len(test)}")

custom_tokens = True
custom_rule_tokens = False
skip_numbers = False
qa = list(map(partial(data_to_qa, custom_tokens=custom_tokens, custom_rule_tokens=custom_rule_tokens,
                      skip_numbers=skip_numbers), train))

examples = datasets.Dataset.from_list(qa)
dataset_qa = examples.train_test_split(test_size=0.2)

tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
if custom_tokens:
    tokenizer.add_special_tokens(
        {"additional_special_tokens": ["[BOR]", "[EOR]", "[BOC]", "[EOC]"]}
    )
if custom_rule_tokens:
    tokenizer.add_special_tokens(
        {"additional_special_tokens": [f"[RULE{rule_n}]" for rule_n in
                                       range(max(len(entry['rules'] for entry in qa)) + 1)]}
    )
model = BertForQuestionAnswering.from_pretrained("bert-base-uncased")
model.resize_token_embeddings(len(tokenizer))

tokenized_dataset = dataset_qa.map(
    partial(tokenize, tokenizer=tokenizer),
    batched=True,
    remove_columns=dataset_qa["train"].column_names,
    num_proc=4,
)

formatted_eval_dataset = dataset_qa["test"].map(format_answers)

task_evaluator = evaluator("question-answering")

def compute_metrics(pred=None):
    eval_results = task_evaluator.compute(
        model_or_pipeline=model,
        data=formatted_eval_dataset,
        metric="squad",
        strategy="simple",
        # n_resamples=9999,
        tokenizer=tokenizer,
        squad_v2_format=False,
    )
    return eval_results

dataset = tokenized_dataset.with_format("torch")

train_loader = DataLoader(dataset["train"], batch_size=16, shuffle=True)
eval_loader = DataLoader(dataset["test"], batch_size=16, shuffle=False)



discarding 0 rules
discarding 0 rules
discarding 0 rules
discarding 21 rules
discarding 0 rules
discarding 21 rules
date len: 17359
data looks like this: 
{'ap_id': 'https://kbin.social/m/pcgaming@lemmy.ca/t/343670/-/comment/1642205', 'applied_rule_n': 2, 'applied_rule_text': 'No Spam or Porn.', 'community': {'actor_id': 'https://lemmy.ca/c/pcgaming', 'description': "For PC gaming news and discussion.\n[PCGamingWiki](https://www.pcgamingwiki.com/wiki/Home)\n\n***Rules:***\n\n0. Be Respectful.\n1. No Spam or Porn.\n2. No Advertising.\n3. No Memes.\n4. No Tech Support.\n5. No questions about buying/building computers.\n6. No game suggestions, friend requests, surveys, or begging.\n7. No Let's Plays, streams, highlight reels/montages, random videos or shorts.\n8. No off-topic posts/comments.\n9. Use the original source, no clickbait titles, no duplicates. \n(Submissions should be from the original source if possible, unless from paywalled or non-english sources.\nIf the title is clickbait

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map (num_proc=4):   0%|          | 0/11109 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/2778 [00:00<?, ? examples/s]

Map:   0%|          | 0/2778 [00:00<?, ? examples/s]

In [67]:

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-6, weight_decay=0.01)
loss = torch.nn.CrossEntropyLoss()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

NUM_EPOCHS = 1
GRADIENT_ACC_STEPS = 1
grads = None
alpha = 0.9
lamb = 0.1
# Train Loop
model.train()
for epoch in range(NUM_EPOCHS):
    train_loss = 0
    steps = 0
    for batch in tqdm.tqdm(
            train_loader, desc=f"Epoch {epoch} | Loss: {train_loss / len(train_loader)}"
    ):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        start_positions = batch["start_positions"].to(device)
        end_positions = batch["end_positions"].to(device)
        outputs = model(
            input_ids,
            attention_mask=attention_mask,
            start_positions=start_positions,
            end_positions=end_positions,
        )
        loss_value = loss(outputs["start_logits"], start_positions) + loss(
            outputs["end_logits"], end_positions
        )
        loss_value = loss_value / GRADIENT_ACC_STEPS
        # with apex.amp.scale_loss(loss_value, optimizer) as scaled_loss:
        #     scaled_loss.backward()
        loss_value.backward()

        if steps % GRADIENT_ACC_STEPS == 0:
            optimizer.step()
            grads = gradfilter_ema(model, grads=grads)
        #
        # optimizer.step()
        train_loss += loss_value.item()
        steps += 1

    print(f"Epoch {epoch} Train Loss: {train_loss / len(train_loader)}")

    # Eval Loop
    model.eval()
    loss_value = 0
    for batch in eval_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        start_positions = batch["start_positions"].to(device)
        end_positions = batch["end_positions"].to(device)

        with torch.no_grad():
            outputs = model(
                input_ids,
                attention_mask=attention_mask,
                start_positions=start_positions,
                end_positions=end_positions,
            )
            loss_value += loss(outputs["start_logits"], start_positions) + loss(
                outputs["end_logits"], end_positions
            )

    print(f"Epoch {epoch} Eval Loss: {loss_value.item() / len(eval_loader)}")

    eval_results = compute_metrics()
    print(eval_results)
    model.train()

Epoch 0 | Loss: 0.0: 100%|██████████| 695/695 [11:20<00:00,  1.02it/s]


Epoch 0 Train Loss: 3.7801668585633204
Epoch 0 Eval Loss: 2.640489775558998


Device set to use cuda:0


{'exact_match': 0.0, 'f1': 25.06511090719515, 'total_time_in_seconds': 33.032730699997046, 'samples_per_second': 84.09840606971825, 'latency_in_seconds': 0.0118908317854561}


In [74]:
outputs['start_logits'].argmax(1).tolist()

[21, 21, 21, 21, 21, 21, 21, 41, 21, 21]

In [68]:
starts_within_gt = list()
ends_within_gt = list()

for s, e, gts, gte in zip(outputs['start_logits'].argmax(1), outputs['end_logits'].argmax(1), start_positions, end_positions):
    s, e, gts, gte = s.item(), e.item(), gts.item(), gte.item()
    starts_within_gt.append(gts<=s<gte)
    ends_within_gt.append(gts<e<=gte)
    
predicted_rules_by_start = np.full(len(starts_within_gt), -1)
predicted_rules_by_end = np.full(len(starts_within_gt), -1)
for rule, rule_boundaries in batch['rule_token_boundaries'].items():
    # print(f'rule {rule}')
    
    for i, s, e, (rs, re )in zip(range(len(outputs['start_logits'])), outputs['start_logits'].argmax(1), outputs['end_logits'].argmax(1),rule_boundaries):
        s, e, rs, re = s.item(), e.item(), rs.item(), re.item()
        if rs<=s<re:
            predicted_rules_by_start[i] = int(rule)
        if rs<e<=re:
            predicted_rules_by_end[i] = int(rule)

In [76]:
def extract_predictions(outputs, batch):
    starts_within_gt = list()
    ends_within_gt = list()
    
    for s, e, gts, gte in zip(outputs['start_logits'].argmax(1), outputs['end_logits'].argmax(1), batch['start_positions'], batch['end_positions']):
        s, e, gts, gte = s.item(), e.item(), gts.item(), gte.item()
        starts_within_gt.append(gts<=s<gte)
        ends_within_gt.append(gts<e<=gte)
        
    predicted_rules_by_start = np.full(len(starts_within_gt), -1)
    predicted_rules_by_end = np.full(len(starts_within_gt), -1)
    for rule, rule_boundaries in batch['rule_token_boundaries'].items():
        # print(f'rule {rule}')
        
        for i, s, e, (rs, re )in zip(range(len(outputs['start_logits'])), outputs['start_logits'].argmax(1), outputs['end_logits'].argmax(1),rule_boundaries):
            s, e, rs, re = s.item(), e.item(), rs.item(), re.item()
            if rs<=s<re:
                predicted_rules_by_start[i] = int(rule)
            if rs<e<=re:
                predicted_rules_by_end[i] = int(rule)
    return starts_within_gt, ends_within_gt, predicted_rules_by_start, predicted_rules_by_end

In [82]:

# Eval Loop
model = model.to(device)
model.eval()
loss_value = 0
predicted_start_logits = list()
predicted_end_logits = list()
true_start_positions = list()
true_end_positions = list()
true_labels = list()

predicted_starts_within_gt=list()
predicted_ends_within_gt=list()
predicted_predicted_rules_by_start=list()
predicted_predicted_rules_by_end=list()


for batch in eval_loader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    start_positions = batch["start_positions"].to(device)
    end_positions = batch["end_positions"].to(device)

    with torch.no_grad():
        outputs = model(
            input_ids,
            attention_mask=attention_mask,
            start_positions=start_positions,
            end_positions=end_positions,
        )
        predicted_start_logits.extend(outputs['start_logits'].argmax(1).tolist())
        predicted_end_logits.extend(outputs['end_logits'].argmax(1).tolist())
        true_end_positions.extend(batch['end_positions'].tolist())
        true_start_positions.extend(batch['start_positions'].tolist())    
        true_labels.extend(batch['answer_rule_number'].tolist())
        
        starts_within_gt, ends_within_gt, predicted_rules_by_start, predicted_rules_by_end = extract_predictions(outputs, batch)
        predicted_starts_within_gt.extend(starts_within_gt)
        predicted_ends_within_gt.extend(ends_within_gt)
        predicted_predicted_rules_by_start.extend(predicted_rules_by_start)
        predicted_predicted_rules_by_end.extend(predicted_rules_by_end)
        
print(f"frac starts within boundary {np.average(predicted_starts_within_gt):.2f}")
print(f"frac ends within boundary {np.average(predicted_ends_within_gt):.2f}")

preds = predicted_predicted_rules_by_start
labels = true_labels
f1 = f1_score(labels, preds, average="macro")
print(f"F1 Score: {f1:.4f}")
print(classification_report(labels, preds, zero_division=0))
print(confusion_matrix(labels, preds))
preds = predicted_predicted_rules_by_end
labels = true_labels
f1 = f1_score(labels, preds, average="macro")
print(f"F1 Score: {f1:.4f}")
print(classification_report(labels, preds))
print(confusion_matrix(labels, preds))

In [28]:
input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
start_positions = batch["start_positions"].to(device)
end_positions = batch["end_positions"].to(device)

with torch.no_grad():
    outputs = model(
        input_ids,
        attention_mask=attention_mask,
        start_positions=start_positions,
        end_positions=end_positions,
    )
    loss_value += loss(outputs["start_logits"], start_positions) + loss(
        outputs["end_logits"], end_positions
    )

In [30]:
outputs['start_logits'].argmax(1)

tensor([21, 21, 24, 41, 21, 21, 21, 21, 21, 21], device='cuda:0')

In [18]:
model = model.to(device)
outputs = model(**inputs.to(device))

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.35 GiB. GPU 0 has a total capacity of 12.00 GiB of which 0 bytes is free. Of the allocated memory 63.26 GiB is allocated by PyTorch, and 346.59 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
starts= outputs.start_logits.argmax().item()

In [19]:
starts

NameError: name 'starts' is not defined

In [ ]:
from datasets import Dataset